In [37]:
import csv
import sklearn

In [38]:
# We can set our K-value here
k = 1

In [39]:
# TASK 1 START: Load the dataset and print all rows
with open('HappinessData-1.csv', mode='r', newline='', encoding='utf-8') as dataset:
    dataset_parser = csv.DictReader(dataset)
    data = list(dataset_parser)

In [40]:
# TASK 2 START: We will reorder the dataset columns
# The target feature (happy/unhappy) will be last column
columns = list(data[0].keys())
new_column_order = columns[1:] + [columns[0]]

# Reorder all datapoints
dataset_reordered = []
for row in data:
    # Create a dictionary following the new order
    reordered_row = {key: row[key] for key in new_column_order}
    dataset_reordered.append(reordered_row)

print("New Order:", list(dataset_reordered[0].keys()))

New Order: ['City Services Availability', 'Housing Cost', 'Quality of schools', 'Community trust in local police', 'Community Maintenance', 'Availability of Community Room ', 'Unhappy/Happy']


In [41]:
# TASK 3 START: We need to handle any missing/NA values
print(f"Pre-clean rows: {len(dataset_reordered)}")

clean_dataset = []
for row in data:
    # If any value is empty, drop row
    if any(value == '' or value is None for value in row.values()):
        continue # Skip this row)
    # Otherwise we still 
    reordered_row = {key: row[key] for key in new_column_order}
    clean_dataset.append(reordered_row)
    
print(f"Post-clean rows: {len(clean_dataset)}")

Pre-clean rows: 140
Post-clean rows: 136


In [42]:
# TASK 4 START: Pearson Correlation

import math
import statistics

# Turn all nums into floats
numeric_data = [[float(val) for val in row.values()] for row in clean_dataset]

# Separate input features from target
X = [row[:-1] for row in numeric_data]
y = [row[-1] for row in numeric_data]

for i in range(len(X[0])):
    # Extract individual feature column
    feature_column = [row[i] for row in X]
    feature_name = new_column_order[i]
    
    # Calculate correlation between this feature and target feature
    r_value = statistics.correlation(feature_column, y)
    
    print(f"{feature_name}: {r_value:.4f}")

City Services Availability: 0.3304
Housing Cost: 0.0245
Quality of schools: 0.1766
Community trust in local police: 0.1234
Community Maintenance: 0.2146
Availability of Community Room : 0.2123


In [170]:
# TASK 5 START: Implement KNN without Scikit
split_pos = int(len(clean_dataset) * 0.8)

# We get our input features X (all features except happy/unhappy)
# We get our target feature Y (happy/unhappy)
X = []
y = []

# Use the Pearson Correlation from before to drop columns
for row in clean_dataset:
    all_features = list(row.values())[:-1]
    filtered_x = [
        float(all_features[0]), # City Services
        float(all_features[2]), # Quality of schools
        float(all_features[4]), # Community Maintenance
        float(all_features[5])  # Availability of Community Room
    ]
    X.append(filtered_x)

    y.append(int(list(row.values())[-1]))

# Make the train/set sets using the split position
X_train, X_test = X[:split_pos], X[split_pos:]
y_train, y_test = y[:split_pos], y[split_pos:]

print(f"Training set: {len(X_train)} rows")
print(f"Testing set: {len(X_test)} rows")

Training set: 108 rows
Testing set: 28 rows


In [84]:
# Normalize features so they all fall between 0-1
mins = [min(column) for column in zip(*X_train)]
maxs = [max(column) for column in zip(*X_train)]

def normalize_dataset(dataset, mins, maxs):
    normalized_data = []
    for row in dataset:
        new_row = []
        for i in range(len(row)):
            denom = float(maxs[i]) - float(mins[i])
            scaled_val = (float(row[i]) - float(mins[i])) / denom
            new_row.append(scaled_val)
        normalized_data.append(new_row)
    return normalized_data

# Apply normalization to both sets
X_train_norm = normalize_dataset(X_train, mins, maxs)
X_test_norm = normalize_dataset(X_test, mins, maxs)

In [85]:
# Function to find k nearest neighbors gets majority vote
def predict_classification(train_X, train_y, test_row, k, distance_func):
    distances = []
    # Calculate distance to every other row (datapoint)
    for i in range(len(train_X)):
        dist = distance_func(test_row, train_X[i])
        distances.append((train_y[i], dist))
    
    # Sort by distance
    distances.sort(key=lambda x: x[1])
    
    # Get the labels of the top k neighbors
    neighbors_labels = [distances[i][0] for i in range(k)]
    
    # Return the most common label (the "mode")
    return statistics.mode(neighbors_labels)

In [161]:
# First distance metric we use is Euclidean
def euclidean(row1, row2):
    distance = 0.0
    for i in range(len(row1)):
        distance += (float(row1[i]) - float(row2[i]))**2
    return math.sqrt(distance)

# Second distance metric is Manhattan
def manhattan(row1, row2):
    distance = 0.0
    for i in range(len(row1)):
        # Sum of absolute differences instead of squared differences
        distance += abs(float(row1[i]) - float(row2[i]))
    return distance

In [166]:
k = 3
predictions = []

for row in X_test_norm:
    result = predict_classification(X_train_norm, y_train, row, k, euclidean)
    predictions.append(result)

# Calculate accuracy
correct = 0
for i in range(len(y_test)):
    if y_test[i] == predictions[i]:
        correct += 1

accuracy = (correct / len(y_test))
print(f"knn Accuracy (k={k}): {accuracy:.2%}")

# Off testing:
# Best k-value is 3 with accuracy of 75%

knn Accuracy (k=3): 75.00%


In [167]:
k = 3
predictions = []

for row in X_test_norm:
    result = predict_classification(X_train_norm, y_train, row, k, manhattan)
    predictions.append(result)

# Calculate accuracy
correct = 0
for i in range(len(y_test)):
    if y_test[i] == predictions[i]:
        correct += 1

accuracy = (correct / len(y_test))
print(f"knn Accuracy (k={k}): {accuracy:.2%}")

# Off testing:
# Best k-value is 3 with accuracy of 75%

knn Accuracy (k=3): 75.00%


In [171]:
# TASK 6 START: Doing KNN using Scikit 

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_sklearn = 5

# Initialize the Scikit-Learn KNN model
knn_model = KNeighborsClassifier(n_neighbors=k_sklearn, metric='manhattan')

# Train the model using the same normalized set as earlier
knn_model.fit(X_train_norm, y_train)

# Make predictions on your testing lists
sklearn_predictions = knn_model.predict(X_test_norm)

# Calculate and print the accuracy
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)
print(f"Scikit-Learn KNN Accuracy Manhattan (k={k_sklearn}): {sklearn_accuracy:.2%}")

# Now we use euclidean distance
knn_model = KNeighborsClassifier(n_neighbors=k_sklearn, metric='euclidean')

# Train the model using the same normalized set as earlier
knn_model.fit(X_train_norm, y_train)

# Make predictions on your testing lists
sklearn_predictions = knn_model.predict(X_test_norm)

# Calculate and print the accuracy
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)
print(f"Scikit-Learn KNN Accuracy Euclidean (k={k_sklearn}): {sklearn_accuracy:.2%}")


# Best for both euclidean & manhattan seems to be:
# k = 5 with accuracy of 64.29%

Scikit-Learn KNN Accuracy Manhattan (k=5): 64.29%
Scikit-Learn KNN Accuracy Euclidean (k=5): 64.29%
